In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv('agri01.csv')

In [8]:
rows=df.shape[0]

In [10]:
df.sample()

,crop,state,district,season,year,min_temperature,max_temperature,avg_temperature,rainfall,humidity,...,potassium,irrigation_type,growth_stage,extreme_event,ndvi,area,previous_year_yield,market_price,profit,yield
3746,Rice,Punjab,Amritsar,Rabi,2020,24.25,33.86,30.72,279.18,71.44,...,74.17,Sprinkler,Maturity,NaN,0.758,6.75,2.43,24246.76,229422.85,1.72


In [12]:
features=[]
for col in df.columns:
    if(df[col].dtype=='float64'):
        features.append(col)
        continue
    elif(df[col].dtype=='object'):
        if(len(df[col].unique())/rows < 0.2):
            features.append(col)
    else:
        if(len(df[col].unique())<rows):
            features.append(col)
print(features)            

['crop', 'state', 'district', 'season', 'year', 'min_temperature', 'max_temperature', 'avg_temperature', 'rainfall', 'humidity', 'soil_type', 'soil_pH', 'organic_matter', 'moisture_content', 'nitrogen', 'phosphorus', 'potassium', 'irrigation_type', 'growth_stage', 'extreme_event', 'ndvi', 'area', 'previous_year_yield', 'market_price', 'profit', 'yield']


In [25]:
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import root_mean_squared_error

In [32]:
models={}

In [33]:
def prediction_model(target):
    if(target in models):
        return models[target]
    if target not in features:
        return f'cant predict {target}'
    else:
        X = df.drop(columns=[target])
        y = df[target]
    
        # Identify categorical columns
        cat_cols = X.select_dtypes(
            include=['object', 'category']
        ).columns.tolist()
    
        # Convert object columns into category
        for col in cat_cols:
            X[col] = X[col].astype('category')
    
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.2,
            random_state=42
        )

        #classification
        if(len(df[target].unique())<10 or df[target].dtype=='object'):
            model = LGBMClassifier(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=-1,
                num_leaves=31,
                random_state=42,
                verbosity=-1
            )
            model.fit(
                X_train,
                y_train,
                categorical_feature=cat_cols
            )
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            models[target]=[model,'accuracy',accuracy]
            return models[target]

        #regression
        else:
            model = LGBMRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=-1,
                num_leaves=31,
                random_state=42,
                verbosity=-1
            )
            model.fit(
                X_train,
                y_train,
                categorical_feature=cat_cols
            )
            y_pred = model.predict(X_test)
            rmse = root_mean_squared_error(y_test, y_pred)
            models[target]=[model,'rmse',rmse]
            return models[target]

In [42]:
prediction_model('district')

[LGBMClassifier(learning_rate=0.05, n_estimators=500, random_state=42,
                verbosity=-1),
 'accuracy',
 0.473]

In [44]:
len(df['state'].unique())

5

In [43]:
models

{'yield': [LGBMRegressor(learning_rate=0.05, n_estimators=500, random_state=42,
                verbosity=-1),
  'rmse',
  0.01585345771437592],
 'irrigation_type': [LGBMClassifier(learning_rate=0.05, n_estimators=500, random_state=42,
                 verbosity=-1),
  'accuracy',
  0.617],
 'state': [LGBMClassifier(learning_rate=0.05, n_estimators=500, random_state=42,
                 verbosity=-1),
  'accuracy',
  1.0],
 'district': [LGBMClassifier(learning_rate=0.05, n_estimators=500, random_state=42,
                 verbosity=-1),
  'accuracy',
  0.473]}